---
## 1. Configuration et Imports

In [2]:
# Imports nécessaires
import os
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Chemins
BASE_PATH = '/home/henintsoa/CFIM'
FINAL_PATH = os.path.join(BASE_PATH, 'final/data')

print("✅ Configuration chargée")
print(f"📂 Chemin des données: {FINAL_PATH}")

✅ Configuration chargée
📂 Chemin des données: /home/henintsoa/CFIM/final/data


---
## 2. Chargement des Données

In [3]:
# Charger les données météo standardisées
print("📂 CHARGEMENT DES DONNÉES MÉTÉO")
print("=" * 60)

df_meteo = pd.read_csv(os.path.join(FINAL_PATH, 'meteo_standardise.csv'))

print(f"\n📊 Dimensions: {df_meteo.shape}")
print(f"📋 Colonnes: {list(df_meteo.columns)}")

# Parser la date
df_meteo['date_parsed'] = pd.to_datetime(df_meteo['date'], format='%d/%m/%Y', errors='coerce')

print(f"\n📅 Période: {df_meteo['date_parsed'].min().date()} à {df_meteo['date_parsed'].max().date()}")
print(f"📍 Zones uniques: {df_meteo['zone'].nunique()}")

display(df_meteo.head())

📂 CHARGEMENT DES DONNÉES MÉTÉO

📊 Dimensions: (1346, 23)
📋 Colonnes: ['date', 'zone', 'vent', 'etat_mer', 'temps', 'vent_vitesse_min', 'vent_vitesse_max', 'vent_rafales', 'vent_direction', 'vent_direction_deg', 'mer_score_min', 'mer_score_max', 'mer_hauteur_min', 'mer_hauteur_max', 'temps_precipitation', 'temps_orage', 'temps_visibilite_reduite', 'temps_clair', 'temps_nuageux', 'temps_intensite', 'temps_score_danger', 'score_risque', 'categorie_risque']

📅 Période: 2019-09-06 à 2022-12-31
📍 Zones uniques: 67


,date,zone,vent,etat_mer,temps,vent_vitesse_min,vent_vitesse_max,vent_rafales,vent_direction,vent_direction_deg,...,temps_precipitation,temps_orage,temps_visibilite_reduite,temps_clair,temps_nuageux,temps_intensite,temps_score_danger,score_risque,categorie_risque,date_parsed
0,06/09/2019,CAP D'AMBRE A MAHANORO,10/15 kt atteignant 20/25 kt au nord d'Antalaha,agitée à forte,Pluies faible a modérée,10.0,15.0,25.0,E,90.0,...,1,0,0,0,0,2,3,49,Élevé (40-60),2019-09-06
1,06/09/2019,MAHANORO AU CAP SAINTE MARIE,05/10 kt devenant progressivement secteur sud ...,agitée à forte,pluies,5.0,10.0,30.0,E,90.0,...,1,0,0,0,0,0,1,43,Élevé (40-60),2019-09-06
2,06/09/2019,CAP D'AMBRE A BESALAMPY,"15/20 kt localement 20 kt au sud de Majunga, v...",Non spécifié,Temps sec,15.0,20.0,20.0,E,90.0,...,0,0,0,1,0,0,0,15,Faible (0-20),2019-09-06
3,06/09/2019,BESALAMPY A MOROMBE,15/20 kt,"agitée a forte, très forte près Morombe dans l...",Temps sec,15.0,20.0,NaN,NaN,NaN,...,0,0,0,1,0,0,0,45,Élevé (40-60),2019-09-06
4,06/09/2019,MOROMBE A CAP SAINTE MARIE,20/25 kt atteignant 30/35 kt entre Morombe et,Non spécifié,Temps partiellement nuageux,20.0,25.0,35.0,E,90.0,...,0,0,0,0,1,0,0,35,Modéré (20-40),2019-09-06


In [4]:
# Charger les données d'incidents géocodés
print("📂 CHARGEMENT DES DONNÉES D'INCIDENTS")
print("=" * 60)

incidents_file = os.path.join(FINAL_PATH, 'incidents_geocodes.csv')

if os.path.exists(incidents_file):
    df_incidents = pd.read_csv(incidents_file)
    print(f"\n✅ Fichier d'incidents géocodés trouvé")
else:
    # Charger les données brutes et les traiter
    print("\n⚠️ Fichier incidents_geocodes.csv non trouvé")
    print("   Chargement des données brutes...")
    df_incidents_raw = pd.read_csv(os.path.join(BASE_PATH, 'csv', 'incidents_maritimes_complets_2017_2022.csv'))
    
    # Filtrer les vrais incidents
    df_incidents = df_incidents_raw[~df_incidents_raw['description'].str.contains('Aucun incident', na=True)].copy()
    
    # Parser la date
    df_incidents['date_incident'] = pd.to_datetime(df_incidents['Date debut'], format='%d/%m/%Y', errors='coerce')
    
    # Renommer les colonnes
    df_incidents.rename(columns={
        'Types': 'type_incident',
        'Region': 'region'
    }, inplace=True)
    
    # Note: zone_cotiere sera assignée plus tard si nécessaire
    df_incidents['zone_cotiere'] = None

print(f"\n📊 Dimensions: {df_incidents.shape}")
print(f"📋 Colonnes: {list(df_incidents.columns)}")

# Parser la date si besoin
if 'date_incident' not in df_incidents.columns:
    df_incidents['date_incident'] = pd.to_datetime(df_incidents['Date debut'], format='%d/%m/%Y', errors='coerce')
else:
    df_incidents['date_incident'] = pd.to_datetime(df_incidents['date_incident'])

print(f"\n📅 Période: {df_incidents['date_incident'].min().date()} à {df_incidents['date_incident'].max().date()}")
print(f"📍 Incidents avec zone: {df_incidents['zone_cotiere'].notna().sum()} / {len(df_incidents)}")

display(df_incidents.head())

📂 CHARGEMENT DES DONNÉES D'INCIDENTS

✅ Fichier d'incidents géocodés trouvé

📊 Dimensions: (269, 17)
📋 Colonnes: ['date_incident', 'annee', 'mois', 'jour', 'jour_semaine', 'saison_cyclonique', 'zone_cotiere', 'region', 'district', 'commune', 'lon_geocode', 'lat_geocode', 'source_geocode', 'type_incident', 'description', 'personnes_concernees', 'deces']

📅 Période: 2017-02-05 à 2022-12-31
📍 Incidents avec zone: 232 / 269


,date_incident,annee,mois,jour,jour_semaine,saison_cyclonique,zone_cotiere,region,district,commune,lon_geocode,lat_geocode,source_geocode,type_incident,description,personnes_concernees,deces
0,2017-02-05,2017,2,5,6,1,CAP D'AMBRE A BESALAMPY,NaN,MADAGASCAR,NaN,46.325890,-15.766938,original,NaN,"A Mahajanga, le cadavre d’un homme retrouvé au...",NaN,NaN
1,2017-02-25,2017,2,25,5,1,CAP D'AMBRE A BESALAMPY,NaN,Katsepy Mahajanga/ Madagascar,NaN,46.289816,-15.737907,original,NaN,Le bac Makuba a coulé 25 Fév,NaN,NaN
2,2017-03-07,2017,3,7,1,1,NaN,NaN,OCEAN INDIEN,NaN,54.740555,-21.401068,original,NaN,"Le vraquier IRIS II (IMO 9286906, dwt 75798, c...",NaN,NaN
3,2017-03-17,2017,3,17,4,1,BESALAMPY A MOROMBE,NaN,madagascar,NaN,43.872715,-18.042964,original,NaN,Un boutre qui a quitté Morondava pour aller à...,NaN,NaN
4,2017-03-31,2017,3,31,4,1,NaN,NaN,OCEAN INDIEN,NaN,68.761516,-20.827302,original,NaN,Un marin philippin âgé de 48 ans a été découve...,NaN,NaN


In [5]:
# Charger les données satellites (optionnel)
print("📂 CHARGEMENT DES DONNÉES SATELLITES (OPTIONNEL)")
print("=" * 60)

swh_file = os.path.join(FINAL_PATH, 'donnees_satellites_swh.csv')

if os.path.exists(swh_file):
    df_swh = pd.read_csv(swh_file)
    df_swh['date'] = pd.to_datetime(df_swh['date'])
    print(f"\n✅ Données satellites chargées")
    print(f"   Période: {df_swh['date'].min().date()} à {df_swh['date'].max().date()}")
    print(f"   Enregistrements: {len(df_swh)}")
    HAS_SATELLITE = True
else:
    print("\n⚠️ Fichier satellites non trouvé (notebook 2.1 non exécuté?)")
    print("   Continuons sans les données satellites")
    df_swh = None
    HAS_SATELLITE = False

📂 CHARGEMENT DES DONNÉES SATELLITES (OPTIONNEL)

✅ Données satellites chargées
   Période: 2017-01-02 à 2018-12-02
   Enregistrements: 288


---
## 3. Analyse de la Correspondance Temporelle

Avant de fusionner, analysons le chevauchement temporel entre les sources de données.

In [6]:
print("📅 ANALYSE DE LA CORRESPONDANCE TEMPORELLE")
print("=" * 60)

# Période météo
meteo_min = df_meteo['date_parsed'].min()
meteo_max = df_meteo['date_parsed'].max()

# Période incidents
incidents_min = df_incidents['date_incident'].min()
incidents_max = df_incidents['date_incident'].max()

print(f"\n📊 Météo: {meteo_min.date()} → {meteo_max.date()}")
print(f"📊 Incidents: {incidents_min.date()} → {incidents_max.date()}")

# Période de chevauchement
overlap_start = max(meteo_min, incidents_min)
overlap_end = min(meteo_max, incidents_max)

if overlap_start < overlap_end:
    overlap_days = (overlap_end - overlap_start).days
    print(f"\n✅ Période commune: {overlap_start.date()} → {overlap_end.date()}")
    print(f"   Durée: {overlap_days} jours (~{overlap_days//30} mois)")
else:
    print("\n❌ ATTENTION: Pas de période commune!")
    print("   Nous allons utiliser les données disponibles")
    overlap_start = meteo_min
    overlap_end = meteo_max

📅 ANALYSE DE LA CORRESPONDANCE TEMPORELLE

📊 Météo: 2019-09-06 → 2022-12-31
📊 Incidents: 2017-02-05 → 2022-12-31

✅ Période commune: 2019-09-06 → 2022-12-31
   Durée: 1212 jours (~40 mois)


In [7]:
# Incidents dans la période commune
df_incidents_overlap = df_incidents[
    (df_incidents['date_incident'] >= overlap_start) &
    (df_incidents['date_incident'] <= overlap_end)
]

print(f"📊 INCIDENTS DANS LA PÉRIODE COMMUNE")
print("=" * 60)
print(f"\nNombre d'incidents: {len(df_incidents_overlap)}")
print(f"Incidents avec zone: {df_incidents_overlap['zone_cotiere'].notna().sum()}")

if len(df_incidents_overlap) > 0:
    print(f"\nTypes d'incidents:")
    print(df_incidents_overlap['type_incident'].value_counts())

📊 INCIDENTS DANS LA PÉRIODE COMMUNE

Nombre d'incidents: 189
Incidents avec zone: 171

Types d'incidents:
type_incident
SAR                            13
ACC_NAUFRAGE                   10
ACC_NOYADE                      8
ACC_PANNE                       4
ACC_ECHOUAGE                    4
ACCIDENT_NOYADE                 3
ACCIDENTS                       3
ACC_INCENDIE                    3
ACC_CHAVIREMENT                 3
ACC                             2
ACC_CADAVRE                     2
ACCIDENT_CADAVRE                2
                                2
ACC_DERIVE                      2
ACC_DISPARITION                 2
ACCIDENT_NAUFRAGE               2
ACCIDENT_DISPARITION            1
ACCIDENT_ECHOUAGE               1
Recif corallien:Marée basse     1
ACCIDENT_DERIVE                 1
ACC_                            1
ASSISTANCE                      1
POLMAR                          1
ACC_COLLISION                   1
Name: count, dtype: int64


---
## 4. Normalisation des Noms de Zones

Les noms de zones peuvent varier légèrement entre les sources. Nous les normalisons pour permettre la fusion.

In [8]:
def normaliser_zone(zone):
    """
    Normalise le nom d'une zone côtière pour la fusion.
    """
    if pd.isna(zone):
        return None
    
    zone = str(zone).upper().strip()
    
    # Supprimer les caractères spéciaux et normaliser les espaces
    zone = zone.replace('\n', ' ').replace('\r', '')
    zone = ' '.join(zone.split())  # Normaliser les espaces multiples
    
    # Standardiser les noms communs
    replacements = {
        "CAP D AMBRE": "CAP D'AMBRE",
        "CAP DAMBRE": "CAP D'AMBRE",
        "SAINTE-MARIE": "SAINTE MARIE",
        "STE MARIE": "SAINTE MARIE",
    }
    
    for old, new in replacements.items():
        zone = zone.replace(old, new)
    
    return zone


# Appliquer la normalisation
df_meteo['zone_norm'] = df_meteo['zone'].apply(normaliser_zone)
df_incidents_overlap = df_incidents_overlap.copy()
df_incidents_overlap['zone_norm'] = df_incidents_overlap['zone_cotiere'].apply(normaliser_zone)

print("📍 ZONES NORMALISÉES")
print("=" * 60)

print("\nZones météo:")
zones_meteo = set(df_meteo['zone_norm'].dropna().unique())
for z in sorted(zones_meteo)[:10]:
    print(f"  • {z}")

print("\nZones incidents:")
zones_incidents = set(df_incidents_overlap['zone_norm'].dropna().unique())
for z in sorted(zones_incidents)[:10]:
    print(f"  • {z}")

# Intersection
zones_communes = zones_meteo.intersection(zones_incidents)
print(f"\n✅ Zones communes: {len(zones_communes)}")

📍 ZONES NORMALISÉES

Zones météo:
  • AMPANIHY A TAOLAGNARO
  • ANTALAHA A FARAFANGANA
  • ANTALAHA A MAHANORO
  • ANTALAHA A MANANJARY
  • ANTALAHA A TAOLAGNARO
  • ANTALAHA A TOAMASINA
  • ANTALAHA AU CAP SAINTE MARIE
  • AVIS AVIS DE GRAND FRAIS ASSOCIE A LA DEPRESSION TROPICALE ENTRE TOAMASINA
  • AVIS AVIS DE GRAND FRAIS ASSOCIE A LA DEPRESSION TROPICALE ENTRE TOAMASINA ET VATOMANDRY
  • AVIS DE GRAND FRAIS ASSOCIE A LA DEPRESSION TROPICALE ENTRE TOAMASINA

Zones incidents:
  • ANTALAHA A TOAMASINA
  • BESALAMPY A MOROMBE
  • CAP D'AMBRE A ANTALAHA
  • CAP D'AMBRE A BESALAMPY
  • CAP D'AMBRE A MAHANORO
  • CAP D'AMBRE A TOAMASINA
  • MAHANORO AU CAP SAINTE MARIE
  • MOROMBE AU CAP SAINTE MARIE
  • NOSY BE ET ENVIRONS

✅ Zones communes: 8


---
## 5. Création du Dataset - NOUVELLE APPROCHE

**Problème identifié**: Les zones météo et les zones incidents ne correspondent pas bien (seulement ~5% de correspondances exactes).

**Solution**: Utiliser une approche basée sur la **DATE** comme clé principale:
- Pour chaque DATE dans la période, on agrège les conditions météo de toutes les zones
- On marque 1 si au moins un incident s'est produit ce jour-là (quelle que soit la zone)
- Cela permet de garder TOUS les incidents au lieu de seulement 6

In [9]:
print("🔄 NOUVELLE APPROCHE: AGRÉGATION PAR DATE")
print("=" * 60)

# Colonnes météo numériques à agréger
colonnes_meteo_num = [
    'vent_vitesse_min', 'vent_vitesse_max', 'vent_rafales', 'vent_direction_deg',
    'mer_score_min', 'mer_score_max', 'mer_hauteur_min', 'mer_hauteur_max',
    'temps_precipitation', 'temps_orage', 'temps_visibilite_reduite',
    'temps_clair', 'temps_nuageux', 'temps_score_danger', 'score_risque'
]

# Vérifier les colonnes disponibles
colonnes_disponibles = [c for c in colonnes_meteo_num if c in df_meteo.columns]
print(f"Colonnes météo disponibles: {len(colonnes_disponibles)}")

# Agrégation par date: on prend les PIRES conditions du jour (max pour risque, min pour visibilité)
agg_dict = {}
for col in colonnes_disponibles:
    if 'min' in col.lower() or 'visibilite' in col.lower() or 'clair' in col.lower():
        agg_dict[col] = ['min', 'mean']  # min et moyenne
    else:
        agg_dict[col] = ['max', 'mean']  # max et moyenne

df_meteo_daily = df_meteo.groupby('date_parsed').agg(agg_dict).reset_index()

# Aplatir les noms de colonnes
df_meteo_daily.columns = ['date'] + [f"{col}_{agg}" for col, agg in df_meteo_daily.columns[1:]]

print(f"\n📊 Météo agrégée par jour:")
print(f"   Lignes: {len(df_meteo_daily)}")
print(f"   Colonnes: {len(df_meteo_daily.columns)}")
print(f"   Période: {df_meteo_daily['date'].min().date()} à {df_meteo_daily['date'].max().date()}")

🔄 NOUVELLE APPROCHE: AGRÉGATION PAR DATE
Colonnes météo disponibles: 15



📊 Météo agrégée par jour:
   Lignes: 495
   Colonnes: 31
   Période: 2019-09-06 à 2022-12-31


---
## 6. Création de la Variable Cible - PAR DATE

Pour chaque jour, on marque 1 si au moins un incident s'est produit ce jour-là.

In [10]:
print("🎯 CRÉATION DE LA VARIABLE CIBLE (PAR DATE)")
print("=" * 60)

# Créer un set des DATES avec incident
dates_avec_incident = set(df_incidents_overlap['date_incident'].dt.date.dropna())

print(f"\n📊 Dates uniques avec au moins un incident: {len(dates_avec_incident)}")

# Assigner la variable cible
df_meteo_daily['date_only'] = df_meteo_daily['date'].dt.date
df_meteo_daily['incident'] = df_meteo_daily['date_only'].apply(
    lambda d: 1 if d in dates_avec_incident else 0
)

print(f"\n📈 Distribution de la variable cible:")
print(df_meteo_daily['incident'].value_counts())
print(f"\nTaux d'incidents: {100*df_meteo_daily['incident'].mean():.2f}%")
print(f"\n✅ {df_meteo_daily['incident'].sum()} jours avec incident sur {len(df_meteo_daily)} jours total")

🎯 CRÉATION DE LA VARIABLE CIBLE (PAR DATE)

📊 Dates uniques avec au moins un incident: 167

📈 Distribution de la variable cible:
incident
0    420
1     75
Name: count, dtype: int64

Taux d'incidents: 15.15%

✅ 75 jours avec incident sur 495 jours total


---
## 7. Ajout des Variables Temporelles

In [11]:
print("📅 AJOUT DES VARIABLES TEMPORELLES")
print("=" * 60)

df_final = df_meteo_daily.copy()

# Variables temporelles
df_final['annee'] = df_final['date'].dt.year
df_final['mois'] = df_final['date'].dt.month
df_final['jour'] = df_final['date'].dt.day
df_final['jour_semaine'] = df_final['date'].dt.dayofweek  # 0=Lundi, 6=Dimanche
df_final['jour_annee'] = df_final['date'].dt.dayofyear

# Saison cyclonique (novembre à avril)
df_final['saison_cyclonique'] = df_final['mois'].isin([11, 12, 1, 2, 3, 4]).astype(int)

# Week-end
df_final['weekend'] = df_final['jour_semaine'].isin([5, 6]).astype(int)

# Variables cycliques pour le mois
df_final['mois_sin'] = np.sin(2 * np.pi * df_final['mois'] / 12)
df_final['mois_cos'] = np.cos(2 * np.pi * df_final['mois'] / 12)

print("\n✅ Variables temporelles ajoutées")
print(f"   Colonnes totales: {len(df_final.columns)}")

📅 AJOUT DES VARIABLES TEMPORELLES

✅ Variables temporelles ajoutées
   Colonnes totales: 42


In [12]:
# Aperçu du dataset
print("📊 APERÇU DU DATASET")
print("=" * 60)

print(f"\nDimensions: {df_final.shape}")
print(f"\nColonnes:")
for i, col in enumerate(df_final.columns):
    print(f"  {i+1}. {col}")

display(df_final.head())

📊 APERÇU DU DATASET

Dimensions: (495, 42)

Colonnes:
  1. date
  2. vent_vitesse_min_min
  3. vent_vitesse_min_mean
  4. vent_vitesse_max_max
  5. vent_vitesse_max_mean
  6. vent_rafales_max
  7. vent_rafales_mean
  8. vent_direction_deg_max
  9. vent_direction_deg_mean
  10. mer_score_min_min
  11. mer_score_min_mean
  12. mer_score_max_max
  13. mer_score_max_mean
  14. mer_hauteur_min_min
  15. mer_hauteur_min_mean
  16. mer_hauteur_max_max
  17. mer_hauteur_max_mean
  18. temps_precipitation_max
  19. temps_precipitation_mean
  20. temps_orage_max
  21. temps_orage_mean
  22. temps_visibilite_reduite_min
  23. temps_visibilite_reduite_mean
  24. temps_clair_min
  25. temps_clair_mean
  26. temps_nuageux_max
  27. temps_nuageux_mean
  28. temps_score_danger_max
  29. temps_score_danger_mean
  30. score_risque_max
  31. score_risque_mean
  32. date_only
  33. incident
  34. annee
  35. mois
  36. jour
  37. jour_semaine
  38. jour_annee
  39. saison_cyclonique
  40. weekend
  41. mo

,date,vent_vitesse_min_min,vent_vitesse_min_mean,vent_vitesse_max_max,vent_vitesse_max_mean,vent_rafales_max,vent_rafales_mean,vent_direction_deg_max,vent_direction_deg_mean,mer_score_min_min,...,incident,annee,mois,jour,jour_semaine,jour_annee,saison_cyclonique,weekend,mois_sin,mois_cos
0,2019-09-06,5.0,13.0,25.0,18.0,35.0,27.50,90.0,90.0,5.0,...,0,2019,9,6,4,249,0,0,-1.0,-1.836970e-16
1,2019-09-07,15.0,18.0,25.0,23.0,30.0,30.00,180.0,144.0,5.0,...,0,2019,9,7,5,250,0,1,-1.0,-1.836970e-16
2,2019-09-09,5.0,9.0,20.0,14.0,30.0,21.25,180.0,117.0,3.0,...,0,2019,9,9,0,252,0,0,-1.0,-1.836970e-16
3,2019-09-10,5.0,11.0,25.0,16.0,30.0,25.00,180.0,108.0,3.0,...,0,2019,9,10,1,253,0,0,-1.0,-1.836970e-16
4,2019-09-13,5.0,7.5,15.0,12.5,25.0,17.50,135.0,82.5,3.0,...,0,2019,9,13,4,256,0,0,-1.0,-1.836970e-16


---
## 8. Gestion des Valeurs Manquantes

In [13]:
print("🔧 GESTION DES VALEURS MANQUANTES")
print("=" * 60)

# Vérifier les valeurs manquantes
missing = df_final.isnull().sum()
missing_cols = missing[missing > 0]

if len(missing_cols) > 0:
    print("\nColonnes avec valeurs manquantes:")
    for col, count in missing_cols.items():
        print(f"  {col}: {count} ({100*count/len(df_final):.1f}%)")
    
    # Imputer par la médiane pour les colonnes numériques
    for col in missing_cols.index:
        if df_final[col].dtype in ['float64', 'int64']:
            median_val = df_final[col].median()
            df_final[col].fillna(median_val, inplace=True)
            print(f"  → {col} imputé avec médiane: {median_val:.2f}")
else:
    print("\n✅ Aucune valeur manquante")

🔧 GESTION DES VALEURS MANQUANTES

Colonnes avec valeurs manquantes:
  vent_vitesse_min_min: 356 (71.9%)
  vent_vitesse_min_mean: 356 (71.9%)
  vent_vitesse_max_max: 356 (71.9%)
  vent_vitesse_max_mean: 356 (71.9%)
  vent_rafales_max: 167 (33.7%)
  vent_rafales_mean: 167 (33.7%)
  vent_direction_deg_max: 11 (2.2%)
  vent_direction_deg_mean: 11 (2.2%)
  mer_score_min_min: 125 (25.3%)
  mer_score_min_mean: 125 (25.3%)
  mer_score_max_max: 125 (25.3%)
  mer_score_max_mean: 125 (25.3%)
  mer_hauteur_min_min: 125 (25.3%)
  mer_hauteur_min_mean: 125 (25.3%)
  mer_hauteur_max_max: 125 (25.3%)
  mer_hauteur_max_mean: 125 (25.3%)
  → vent_vitesse_min_min imputé avec médiane: 5.00
  → vent_vitesse_min_mean imputé avec médiane: 8.75
  → vent_vitesse_max_max imputé avec médiane: 20.00
  → vent_vitesse_max_mean imputé avec médiane: 14.00
  → vent_rafales_max imputé avec médiane: 25.00
  → vent_rafales_mean imputé avec médiane: 22.50
  → vent_direction_deg_max imputé avec médiane: 135.00
  → vent_dir

---
## 9. Création des Features Dérivées

In [14]:
print("🧮 CRÉATION DE FEATURES DÉRIVÉES")
print("=" * 60)

# Amplitude du vent (variation journalière)
if 'vent_vitesse_max_max' in df_final.columns and 'vent_vitesse_min_min' in df_final.columns:
    df_final['vent_amplitude'] = df_final['vent_vitesse_max_max'] - df_final['vent_vitesse_min_min']

# Vent moyen
if 'vent_vitesse_max_mean' in df_final.columns and 'vent_vitesse_min_mean' in df_final.columns:
    df_final['vent_moyen'] = (df_final['vent_vitesse_max_mean'] + df_final['vent_vitesse_min_mean']) / 2

# Rafales fortes (indicateur binaire si rafales > 30 kt)
if 'vent_rafales_max' in df_final.columns:
    df_final['rafales_fortes'] = (df_final['vent_rafales_max'] >= 30).astype(int)

# Mer hauteur moyenne
if 'mer_hauteur_max_max' in df_final.columns and 'mer_hauteur_min_min' in df_final.columns:
    df_final['mer_hauteur_moy'] = (df_final['mer_hauteur_max_max'] + df_final['mer_hauteur_min_min']) / 2
    df_final['mer_amplitude'] = df_final['mer_hauteur_max_max'] - df_final['mer_hauteur_min_min']

# Score de conditions défavorables
conditions_cols = [c for c in df_final.columns if 'precipitation' in c or 'orage' in c or 'visibilite' in c]
if conditions_cols:
    df_final['nb_conditions_defavorables'] = df_final[conditions_cols].sum(axis=1)

# Score météo combiné
score_cols = [c for c in df_final.columns if 'score_risque' in c]
if score_cols:
    df_final['score_meteo_max'] = df_final[score_cols].max(axis=1)

print(f"\n✅ Features dérivées créées")
print(f"   Colonnes totales: {len(df_final.columns)}")

🧮 CRÉATION DE FEATURES DÉRIVÉES

✅ Features dérivées créées
   Colonnes totales: 49


---
## 10. Split Train/Test et Export

In [15]:
from sklearn.model_selection import train_test_split

print("📦 PRÉPARATION DES DATASETS FINAUX")
print("=" * 60)

# Colonnes à exclure des features
cols_to_exclude = ['date', 'date_only', 'incident']

# Sélectionner les features
feature_cols = [c for c in df_final.columns if c not in cols_to_exclude]
print(f"\nFeatures sélectionnées: {len(feature_cols)}")

# Préparer X et y
X = df_final[feature_cols]
y = df_final['incident']

# Split stratifié
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\n📊 SPLIT TRAIN/TEST:")
print(f"   Train: {len(X_train)} échantillons")
print(f"      - Incidents: {y_train.sum()} ({100*y_train.mean():.1f}%)")
print(f"      - Non-incidents: {(y_train == 0).sum()} ({100*(1-y_train.mean()):.1f}%)")
print(f"   Test: {len(X_test)} échantillons")
print(f"      - Incidents: {y_test.sum()} ({100*y_test.mean():.1f}%)")
print(f"      - Non-incidents: {(y_test == 0).sum()} ({100*(1-y_test.mean()):.1f}%)")

📦 PRÉPARATION DES DATASETS FINAUX

Features sélectionnées: 46

📊 SPLIT TRAIN/TEST:
   Train: 396 échantillons
      - Incidents: 60 (15.2%)
      - Non-incidents: 336 (84.8%)
   Test: 99 échantillons
      - Incidents: 15 (15.2%)
      - Non-incidents: 84 (84.8%)


---
## 11. Export des Datasets

In [16]:
print("💾 EXPORT DES DATASETS")
print("=" * 60)

# Créer les DataFrames pour l'export
train_df = X_train.copy()
train_df['incident'] = y_train.values

test_df = X_test.copy()
test_df['incident'] = y_test.values

# Exporter
train_df.to_csv(os.path.join(FINAL_PATH, 'dataset_train.csv'), index=False)
test_df.to_csv(os.path.join(FINAL_PATH, 'dataset_test.csv'), index=False)

# Exporter aussi le dataset complet
df_final.to_csv(os.path.join(FINAL_PATH, 'dataset_entrainement.csv'), index=False)

print("\n✅ Fichiers exportés:")
print(f"   • dataset_train.csv ({len(train_df)} lignes)")
print(f"   • dataset_test.csv ({len(test_df)} lignes)")
print(f"   • dataset_entrainement.csv ({len(df_final)} lignes)")

💾 EXPORT DES DATASETS

✅ Fichiers exportés:
   • dataset_train.csv (396 lignes)
   • dataset_test.csv (99 lignes)
   • dataset_entrainement.csv (495 lignes)


In [17]:
print("=" * 60)
print("🎉 CRÉATION DU DATASET TERMINÉE!")
print("=" * 60)
print(f"""
📊 RÉSUMÉ:
   - Approche: Agrégation des conditions météo PAR JOUR
   - Total jours: {len(df_final)}
   - Jours avec incident: {df_final['incident'].sum()} ({100*df_final['incident'].mean():.1f}%)
   - Features météo: conditions max/mean du jour
   - Features temporelles: mois, jour, saison cyclonique, etc.

📁 FICHIERS CRÉÉS:
   • dataset_train.csv - pour l'entraînement
   • dataset_test.csv - pour l'évaluation
   • dataset_entrainement.csv - dataset complet

➡️ PROCHAINE ÉTAPE: Exécuter le notebook model.ipynb
""")

🎉 CRÉATION DU DATASET TERMINÉE!

📊 RÉSUMÉ:
   - Approche: Agrégation des conditions météo PAR JOUR
   - Total jours: 495
   - Jours avec incident: 75 (15.2%)
   - Features météo: conditions max/mean du jour
   - Features temporelles: mois, jour, saison cyclonique, etc.

📁 FICHIERS CRÉÉS:
   • dataset_train.csv - pour l'entraînement
   • dataset_test.csv - pour l'évaluation
   • dataset_entrainement.csv - dataset complet

➡️ PROCHAINE ÉTAPE: Exécuter le notebook model.ipynb

